# Phase 2: Reacher-hard Experiments

**Notebook:** `06_reacher_hard_experiments.ipynb`  
**Phase:** 2 - DMControl Suite  
**Author:** Saurabh Jalendra  

## Objectives
1. Run all 5 quantum-inspired approaches on Reacher-hard (DMControl)
2. Collect multi-seed results (5 seeds each)
3. Evaluate test set performance and long-horizon prediction
4. Save results to experiments/results/phase2/reacher_hard/


In [1]:
# GPU Verification - Run this first!
import sys
import torch
print(f"Python executable: {sys.executable}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print()
    print("=" * 60)
    print("WARNING: GPU NOT AVAILABLE!")
    print("You are likely using the wrong kernel.")
    print("Go to: Kernel > Change Kernel > Python (quantum-rl-venv GPU)")
    print("Or launch Jupyter from the venv:")
    print(r"  venv\Scripts\jupyter lab")
    print("=" * 60)
    raise RuntimeError("GPU not available - wrong kernel! See instructions above.")


Python executable: d:\Git Repos\Quantum-Enhanced-Simulation-Learning-for-Reinforcement-Learning\venv\Scripts\python.exe
PyTorch version: 2.10.0.dev20251124+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
CUDA version: 12.8


In [2]:
"""
Cell: Imports and Configuration
Purpose: Set up environment, imports, and experiment configuration
"""
import sys
import os
import json
import time
import traceback
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, NamedTuple
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# DMControl Suite
from dm_control import suite

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Import quantum-inspired components
from quantum_inspired import (
    QuantumTunnelingOptimizer,
    SuperpositionReplayBuffer,
    EntanglementLayer,
    InterferenceEnsemble,
)

# Device setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configuration
OBS_DIM = 6
ACTION_DIM = 2
STOCH_DIM = 64
DETER_DIM = 512
HIDDEN_DIM = 512
STATE_DIM = DETER_DIM + STOCH_DIM

NUM_STEPS = 10000
BATCH_SIZE = 32
SEQ_LEN = 20
LEARNING_RATE = 3e-4
KL_WEIGHT = 1.0
GRAD_CLIP = 100.0
NUM_EPISODES = 200
NUM_ENSEMBLE_MODELS = 5
INTERFERENCE_STRENGTH = 0.7
EXPERIMENT_SEEDS = [42, 123, 456, 789, 1024]

APPROACHES = ["baseline", "quantum_tunneling", "superposition", "entanglement", "interference_ensemble"]

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results" / "phase2" / "reacher_hard"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration: obs_dim={OBS_DIM}, action_dim={ACTION_DIM}")
print(f"Architecture: stoch={STOCH_DIM}, deter={DETER_DIM}, hidden={HIDDEN_DIM}")
print(f"Training: steps={NUM_STEPS}, batch={BATCH_SIZE}, seq_len={SEQ_LEN}, lr={LEARNING_RATE}")
print(f"Episodes: {NUM_EPISODES} (DMControl collects more for better coverage)")
print(f"Seeds: {EXPERIMENT_SEEDS}")
print(f"Results directory: {RESULTS_DIR}")

Device: cuda
GPU: NVIDIA GeForce RTX 5090
Configuration: obs_dim=6, action_dim=2
Architecture: stoch=64, deter=512, hidden=512
Training: steps=10000, batch=32, seq_len=20, lr=0.0003
Episodes: 200 (DMControl collects more for better coverage)
Seeds: [42, 123, 456, 789, 1024]
Results directory: d:\Git Repos\Quantum-Enhanced-Simulation-Learning-for-Reinforcement-Learning\experiments\results\phase2\reacher_hard


---
## Data Collection

Collect Reacher-hard episodes from DMControl Suite using a random policy.
Reacher-hard has obs_dim=6 (position, to_target, velocity) and action_dim=2.


In [3]:
"""
Cell: Data Collection and Replay Buffer
Purpose: Collect Reacher-hard episodes using DMControl Suite
"""

def set_seed(seed):
    """Set all random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def flatten_obs(obs):
    """Flatten DMControl observation dictionary to array."""
    parts = []
    for key in sorted(obs.keys()):
        val = np.asarray(obs[key], dtype=np.float32).flatten()
        parts.append(val)
    return np.concatenate(parts)


def collect_episodes(domain, task, num_episodes, seed):
    """Collect episodes from DMControl environment using random policy.

    Uses a local RandomState seeded per call to ensure deterministic data
    collection regardless of global NumPy random state.

    Returns list of dicts with keys: 'obs', 'actions', 'rewards', 'dones'.
    """
    rng = np.random.RandomState(seed)
    env = suite.load(domain, task, task_kwargs={"random": seed})
    episodes = []
    for i in range(num_episodes):
        timestep = env.reset()
        obs_list, act_list, rew_list, done_list = [], [], [], []
        while not timestep.last():
            obs = flatten_obs(timestep.observation)
            obs_list.append(obs)
            action = rng.uniform(
                -1, 1, size=env.action_spec().shape
            ).astype(np.float32)
            timestep = env.step(action)
            act_list.append(action)
            rew_list.append(float(timestep.reward))
            done_list.append(float(timestep.last()))
        if len(obs_list) > 0:
            episodes.append({
                "obs": np.array(obs_list, dtype=np.float32),
                "actions": np.array(act_list, dtype=np.float32),
                "rewards": np.array(rew_list, dtype=np.float32),
                "dones": np.array(done_list, dtype=np.float32),
            })
    return episodes


class ReplayBuffer:
    """Simple episode replay buffer.

    .add(episode_dict) stores an episode dict.
    .sample(batch_size, seq_len) returns tuple (obs, actions, rewards, dones)
    of numpy arrays with shape (batch_size, seq_len, ...).
    """

    def __init__(self, capacity=10000):
        self.episodes = []
        self.capacity = capacity

    def add(self, episode):
        if len(self.episodes) >= self.capacity:
            self.episodes.pop(0)
        self.episodes.append(episode)

    def sample(self, batch_size, seq_len):
        obs_b, act_b, rew_b, done_b = [], [], [], []
        for _ in range(batch_size):
            ep = self.episodes[np.random.randint(len(self.episodes))]
            L = len(ep["obs"])
            if L <= seq_len:
                pad = seq_len - L
                obs = np.pad(ep["obs"], ((0, pad), (0, 0)), mode="edge")
                act = np.pad(ep["actions"], ((0, pad), (0, 0)), mode="edge")
                rew = np.pad(ep["rewards"], (0, pad), mode="edge")
                done = np.pad(ep["dones"], (0, pad), mode="edge")
            else:
                s = np.random.randint(0, L - seq_len)
                obs = ep["obs"][s : s + seq_len]
                act = ep["actions"][s : s + seq_len]
                rew = ep["rewards"][s : s + seq_len]
                done = ep["dones"][s : s + seq_len]
            obs_b.append(obs)
            act_b.append(act)
            rew_b.append(rew)
            done_b.append(done)
        return np.array(obs_b), np.array(act_b), np.array(rew_b), np.array(done_b)

    def __len__(self):
        return len(self.episodes)


print("Data collection utilities defined.")
print("Testing DMControl Reacher-hard...")
test_eps = collect_episodes("reacher", "hard", 3, seed=42)
print(f"  Collected {len(test_eps)} test episodes")
if test_eps:
    print(f"  Obs shape: {test_eps[0]['obs'].shape}")
    print(f"  Action shape: {test_eps[0]['actions'].shape}")
    print(f"  Mean episode length: {np.mean([len(ep['obs']) for ep in test_eps]):.0f}")

Data collection utilities defined.
Testing DMControl Reacher-hard...
  Collected 3 test episodes
  Obs shape: (1000, 6)
  Action shape: (1000, 2)
  Mean episode length: 1000


---
## RSSM World Model Architecture

Standard RSSM architecture matching the project specification:  
- Encoder: obs_dim -> 512 -> 512 -> 512  
- Decoder: state_dim -> 512 -> 512 -> obs_dim  
- Reward predictor: state_dim -> 512 -> 512 -> 1  
- Continue predictor: state_dim -> 512 -> 512 -> 1  

This base model is used by all approaches. The `forward` method returns `(predictions, states_dict)` where
`states_dict` contains `'deter'`, `'stoch'`, `'priors'`, and `'posteriors'` -- matching the format
expected by `InterferenceEnsemble`.

In [4]:
"""
Cell: RSSM World Model Architecture
Purpose: Define the base world model used by all approaches
"""

class RSSMState(NamedTuple):
    deter: torch.Tensor
    stoch: torch.Tensor

    @property
    def combined(self):
        return torch.cat([self.deter, self.stoch], dim=-1)


class BaseWorldModel(nn.Module):
    """Standard RSSM world model matching the project architecture spec.

    forward() returns 2-tuple: (predictions, states_dict)
    where states_dict has keys: 'deter', 'stoch', 'priors', 'posteriors'.

    initial_state(batch_size, device) takes device as positional parameter.
    """

    def __init__(self, obs_dim=OBS_DIM, action_dim=ACTION_DIM,
                 stoch_dim=STOCH_DIM, deter_dim=DETER_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.stoch_dim = stoch_dim
        self.deter_dim = deter_dim
        self.hidden_dim = hidden_dim
        self.state_dim = stoch_dim + deter_dim

        # Encoder: obs_dim -> 512 -> 512 -> 512
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # Input projection: stoch + action -> hidden
        self.input_proj = nn.Sequential(
            nn.Linear(stoch_dim + action_dim, hidden_dim), nn.ELU()
        )

        # GRU dynamics
        self.gru = nn.GRUCell(hidden_dim, deter_dim)

        # Prior: deter -> stoch*2
        self.prior_net = nn.Sequential(
            nn.Linear(deter_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, stoch_dim * 2),
        )

        # Posterior: deter + embed -> stoch*2
        self.posterior_net = nn.Sequential(
            nn.Linear(deter_dim + hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, stoch_dim * 2),
        )

        # Decoder: state -> obs
        self.decoder = nn.Sequential(
            nn.Linear(self.state_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, obs_dim),
        )

        # Reward predictor
        self.reward_pred = nn.Sequential(
            nn.Linear(self.state_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, 1),
        )

        # Continue predictor
        self.continue_pred = nn.Sequential(
            nn.Linear(self.state_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, 1),
        )

    def initial_state(self, batch_size, device):
        """Create initial RSSM state. Takes device as positional parameter."""
        return RSSMState(
            deter=torch.zeros(batch_size, self.deter_dim, device=device),
            stoch=torch.zeros(batch_size, self.stoch_dim, device=device),
        )

    def _get_dist(self, stats):
        mean, log_std = stats.chunk(2, dim=-1)
        std = F.softplus(log_std) + 0.1
        return torch.distributions.Normal(mean, std)

    def observe(self, obs, action, state):
        embed = self.encoder(obs)
        x = self.input_proj(torch.cat([state.stoch, action], dim=-1))
        deter = self.gru(x, state.deter)
        posterior_stats = self.posterior_net(torch.cat([deter, embed], dim=-1))
        posterior = self._get_dist(posterior_stats)
        stoch = posterior.rsample()
        prior_stats = self.prior_net(deter)
        prior = self._get_dist(prior_stats)
        return RSSMState(deter, stoch), prior, posterior

    def decode(self, state):
        return self.decoder(state.combined)

    def forward(self, obs_seq, action_seq):
        """Forward pass through sequence.

        Returns
        -------
        predictions : Tensor (batch, seq_len, obs_dim)
        states_dict : dict with keys 'deter', 'stoch', 'priors', 'posteriors'
        """
        batch_size, seq_len = obs_seq.shape[:2]
        device = obs_seq.device
        state = self.initial_state(batch_size, device)

        recon_obs, all_deter, all_stoch = [], [], []
        priors, posteriors = [], []

        for t in range(seq_len):
            state, prior, posterior = self.observe(
                obs_seq[:, t], action_seq[:, t], state
            )
            recon_obs.append(self.decode(state))
            all_deter.append(state.deter)
            all_stoch.append(state.stoch)
            priors.append(prior)
            posteriors.append(posterior)

        predictions = torch.stack(recon_obs, dim=1)
        states_dict = {
            "deter": torch.stack(all_deter, dim=1),
            "stoch": torch.stack(all_stoch, dim=1),
            "priors": priors,
            "posteriors": posteriors,
        }
        return predictions, states_dict


# Test model creation
test_model = BaseWorldModel().to(DEVICE)
num_params = sum(p.numel() for p in test_model.parameters())
print(f"BaseWorldModel created: {num_params:,} parameters")
del test_model

BaseWorldModel created: 4,736,264 parameters


---
## Quantum-Inspired Model Variants

- **EntanglementWorldModel**: BaseWorldModel with EntanglementLayer in the encoder
- **InterferenceWorldModel**: Wraps 5 BaseWorldModels via InterferenceEnsemble

In [5]:
"""
Cell: Quantum-Inspired Model Variants
Purpose: Define EntanglementWorldModel and InterferenceWorldModel
"""

class EntanglementWorldModel(BaseWorldModel):
    """RSSM with EntanglementLayer inserted into the encoder."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Replace encoder with entanglement-enhanced version
        hidden_dim = kwargs.get("hidden_dim", HIDDEN_DIM)
        obs_dim = kwargs.get("obs_dim", OBS_DIM)
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ELU(),
            EntanglementLayer(dim=hidden_dim),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )


class InterferenceWorldModel(nn.Module):
    """Wrapper around InterferenceEnsemble for the interference approach."""

    def __init__(self, base_seed=None):
        super().__init__()
        self.ensemble = InterferenceEnsemble(
            model_class=BaseWorldModel,
            num_models=NUM_ENSEMBLE_MODELS,
            interference_strength=INTERFERENCE_STRENGTH,
            uncertainty_method="disagreement",
            base_seed=base_seed,
            obs_dim=OBS_DIM,
            action_dim=ACTION_DIM,
            stoch_dim=STOCH_DIM,
            deter_dim=DETER_DIM,
            hidden_dim=HIDDEN_DIM,
        )

    def forward(self, obs_seq, action_seq):
        return self.ensemble(obs_seq, action_seq, return_all=True)


def create_model(approach, seed=42):
    """Factory function to create the correct model for each approach."""
    if approach == "entanglement":
        model = EntanglementWorldModel(
            obs_dim=OBS_DIM, action_dim=ACTION_DIM,
            stoch_dim=STOCH_DIM, deter_dim=DETER_DIM, hidden_dim=HIDDEN_DIM,
        )
    elif approach == "interference_ensemble":
        model = InterferenceWorldModel(base_seed=seed)
    else:
        # baseline, quantum_tunneling, superposition all use BaseWorldModel
        model = BaseWorldModel(
            obs_dim=OBS_DIM, action_dim=ACTION_DIM,
            stoch_dim=STOCH_DIM, deter_dim=DETER_DIM, hidden_dim=HIDDEN_DIM,
        )
    return model.to(DEVICE)


# Verify all model variants
for approach in APPROACHES:
    m = create_model(approach)
    p = sum(p.numel() for p in m.parameters())
    print(f"  {approach}: {p:,} params")
    del m
torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

  baseline: 4,736,264 params
  quantum_tunneling: 4,736,264 params
  superposition: 4,736,264 params
  entanglement: 5,262,216 params
  interference_ensemble: 23,681,326 params


---
## Loss Functions

Two loss functions:
- `compute_loss`: For single-model approaches (baseline, tunneling, superposition, entanglement)
- `compute_ensemble_loss`: For interference ensemble approach

In [6]:
"""
Cell: Loss Functions
Purpose: Define loss computation for standard and ensemble training
"""

def compute_loss(model, obs_seq, action_seq, reward_seq, kl_weight=KL_WEIGHT):
    """Standard RSSM loss for single-model approaches.

    Uses 2-tuple return from forward(): (predictions, states_dict).
    states_dict has keys: 'deter', 'stoch', 'priors', 'posteriors'.
    """
    predictions, states = model(obs_seq, action_seq)
    recon_loss = F.mse_loss(predictions, obs_seq)

    kl_losses = []
    for prior, posterior in zip(states["priors"], states["posteriors"]):
        kl = torch.distributions.kl_divergence(posterior, prior).sum(-1).mean()
        kl_losses.append(kl)
    kl_loss = (
        torch.stack(kl_losses).mean()
        if kl_losses
        else torch.tensor(0.0, device=obs_seq.device)
    )

    combined_states = torch.cat([states["deter"], states["stoch"]], dim=-1)
    reward_pred = model.reward_pred(combined_states)
    reward_loss = F.mse_loss(reward_pred.squeeze(-1), reward_seq)

    total = recon_loss + kl_weight * kl_loss + reward_loss
    return {"total": total, "recon": recon_loss, "kl": kl_loss, "reward": reward_loss}


def compute_ensemble_loss(ensemble_model, obs_seq, action_seq, reward_seq, kl_weight=KL_WEIGHT):
    """Loss for InterferenceEnsemble approach.

    Combines individual model losses, ensemble combined loss,
    KL divergence, and a diversity bonus.
    """
    combined_pred, states = ensemble_model(obs_seq, action_seq)
    all_predictions = states.get("all_predictions", None)

    combined_recon_loss = F.mse_loss(combined_pred, obs_seq)

    individual_loss = torch.tensor(0.0, device=obs_seq.device)
    if all_predictions is not None:
        individual_losses = [
            F.mse_loss(pred, obs_seq) for pred in all_predictions
        ]
        individual_loss = torch.stack(individual_losses).mean()

    kl_losses = []
    if "all_states" in states:
        for model_states in states["all_states"]:
            if "priors" in model_states and "posteriors" in model_states:
                for prior, posterior in zip(
                    model_states["priors"], model_states["posteriors"]
                ):
                    kl = (
                        torch.distributions.kl_divergence(posterior, prior)
                        .sum(-1)
                        .mean()
                    )
                    kl_losses.append(kl)
    kl_loss = (
        torch.stack(kl_losses).mean()
        if kl_losses
        else torch.tensor(0.0, device=obs_seq.device)
    )

    diversity = torch.tensor(0.0, device=obs_seq.device)
    if all_predictions is not None:
        mean_pred = all_predictions.mean(dim=0)
        diversity = ((all_predictions - mean_pred) ** 2).mean()

    total = (
        0.5 * combined_recon_loss
        + 0.5 * individual_loss
        + kl_weight * kl_loss
        - 0.01 * diversity
    )
    return {
        "total": total,
        "combined_recon": combined_recon_loss,
        "individual_recon": individual_loss,
        "kl": kl_loss,
        "diversity": diversity,
    }


print("Loss functions defined.")

Loss functions defined.


---
## Training Functions

- `train_single_model`: Trains baseline, quantum_tunneling, superposition, and entanglement approaches
- `train_ensemble`: Trains the interference_ensemble approach

In [7]:
"""
Cell: Training Functions
Purpose: Training loops for single-model and ensemble approaches
"""

def train_single_model(model, buffer, approach, seed, num_steps=NUM_STEPS):
    """Train a single-model approach (baseline, tunneling, superposition, entanglement).

    Parameters
    ----------
    model : BaseWorldModel or EntanglementWorldModel
    buffer : ReplayBuffer or SuperpositionReplayBuffer
    approach : str
        Approach name (affects optimizer and buffer sampling)
    seed : int
        Random seed for this run
    num_steps : int
        Number of gradient steps

    Returns
    -------
    pd.DataFrame with training history
    """
    # Select optimizer
    if approach == "quantum_tunneling":
        optimizer = QuantumTunnelingOptimizer(
            model.parameters(),
            lr=LEARNING_RATE,
            tunneling_strength=0.001,
            annealing_rate=0.9999,
            tunneling_frequency=100,
            min_tunneling=1e-8,
            stuck_threshold=500,
            base_optimizer="adamw",
        )
    else:
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    use_superposition_buffer = approach == "superposition"
    history = []

    for step in range(num_steps):
        model.train()

        if use_superposition_buffer:
            batch = buffer.sample(batch_size=BATCH_SIZE, seq_len=SEQ_LEN)
            obs = batch["obs"].to(DEVICE)
            actions = batch["actions"].to(DEVICE)
            rewards = batch["rewards"].to(DEVICE)
        else:
            obs_np, act_np, rew_np, _ = buffer.sample(BATCH_SIZE, SEQ_LEN)
            obs = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)
            actions = torch.tensor(act_np, dtype=torch.float32, device=DEVICE)
            rewards = torch.tensor(rew_np, dtype=torch.float32, device=DEVICE)

        optimizer.zero_grad()
        losses = compute_loss(model, obs, actions, rewards)
        losses["total"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        if approach == "quantum_tunneling":
            optimizer.step(losses["total"].item())
        else:
            optimizer.step()

        history.append({
            "step": step,
            "total": losses["total"].item(),
            "recon": losses["recon"].item(),
            "kl": losses["kl"].item(),
            "reward": losses["reward"].item(),
        })

        if step % 1000 == 0:
            print(
                f"    Step {step}/{num_steps}: total={losses['total'].item():.4f} "
                f"recon={losses['recon'].item():.4f} kl={losses['kl'].item():.4f} "
                f"reward={losses['reward'].item():.4f}"
            )

    return pd.DataFrame(history)


def train_interference_ensemble(model, buffer, num_steps=NUM_STEPS):
    """Train interference ensemble approach.

    Parameters
    ----------
    model : InterferenceWorldModel
    buffer : ReplayBuffer
    num_steps : int

    Returns
    -------
    pd.DataFrame with training history
    """
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    history = []

    for step in range(num_steps):
        model.train()
        obs_np, act_np, rew_np, _ = buffer.sample(BATCH_SIZE, SEQ_LEN)
        obs = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)
        actions = torch.tensor(act_np, dtype=torch.float32, device=DEVICE)
        rewards = torch.tensor(rew_np, dtype=torch.float32, device=DEVICE)

        optimizer.zero_grad()
        losses = compute_ensemble_loss(model, obs, actions, rewards)
        losses["total"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        history.append({
            "step": step,
            "total": losses["total"].item(),
            "combined_recon": losses["combined_recon"].item(),
            "individual_recon": losses["individual_recon"].item(),
            "kl": losses["kl"].item(),
            "diversity": losses["diversity"].item(),
        })

        if step % 1000 == 0:
            print(
                f"    Step {step}/{num_steps}: total={losses['total'].item():.4f} "
                f"combined={losses['combined_recon'].item():.4f} "
                f"kl={losses['kl'].item():.4f}"
            )

    return pd.DataFrame(history)


print("Training functions defined.")

Training functions defined.


---
## Evaluation Functions

Three evaluations per trained model:
1. **Train set evaluation**: Reconstruction MSE on training episodes
2. **Test set evaluation**: Reconstruction MSE on held-out episodes
3. **Long-horizon prediction**: Imagination accuracy at horizons [5, 10, 15, 20]

In [8]:
"""
Cell: Evaluation Functions
Purpose: Evaluate trained models on train set, test set, and long-horizon prediction
"""

def evaluate_model(model, buffer, approach, num_eval_batches=10):
    """Evaluate a model on data from the buffer.

    Parameters
    ----------
    model : nn.Module
        Trained model
    buffer : ReplayBuffer
        Buffer with evaluation episodes
    approach : str
        Approach name (affects state access for reward prediction)
    num_eval_batches : int
        Number of batches to average over

    Returns
    -------
    dict with 'obs_mse' and 'reward_mse'
    """
    total_mse = 0.0
    total_reward_mse = 0.0
    count = 0

    with torch.no_grad():
        for _ in range(num_eval_batches):
            obs_np, act_np, rew_np, _ = buffer.sample(BATCH_SIZE, SEQ_LEN)
            obs = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)
            actions = torch.tensor(act_np, dtype=torch.float32, device=DEVICE)
            rewards = torch.tensor(rew_np, dtype=torch.float32, device=DEVICE)

            model.eval()
            if approach == "interference_ensemble":
                pred, states = model(obs, actions)
            else:
                pred, states = model(obs, actions)

            mse = F.mse_loss(pred, obs).item()
            total_mse += mse

            # Reward MSE
            if approach == "interference_ensemble":
                first_states = (
                    states["all_states"][0]
                    if "all_states" in states
                    else states
                )
                combined = torch.cat(
                    [first_states["deter"], first_states["stoch"]], dim=-1
                )
                rpred = model.ensemble.models[0].reward_pred(combined).squeeze(-1)
            else:
                combined = torch.cat(
                    [states["deter"], states["stoch"]], dim=-1
                )
                rpred = model.reward_pred(combined).squeeze(-1)

            reward_mse = F.mse_loss(rpred, rewards).item()
            total_reward_mse += reward_mse
            count += 1

    return {
        "obs_mse": total_mse / count,
        "reward_mse": total_reward_mse / count,
    }


def evaluate_long_horizon(model, buffer, approach, horizons=None):
    """Evaluate prediction quality at different horizons.

    Parameters
    ----------
    model : nn.Module
    buffer : ReplayBuffer
    approach : str
    horizons : list of int

    Returns
    -------
    dict mapping horizon (int) -> MSE (float)
    """
    if horizons is None:
        horizons = [5, 10, 15, 20]
    max_horizon = max(horizons)

    results = {}
    with torch.no_grad():
        obs_np, act_np, _, _ = buffer.sample(BATCH_SIZE, max_horizon)
        obs = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)
        actions = torch.tensor(act_np, dtype=torch.float32, device=DEVICE)

        if approach == "interference_ensemble":
            pred, _ = model(obs, actions)
        else:
            model.eval()
            pred, _ = model(obs, actions)

        for h in horizons:
            mse = F.mse_loss(pred[:, :h], obs[:, :h]).item()
            results[h] = mse

    return results


print("Evaluation functions defined.")

Evaluation functions defined.


---
## Run All Experiments

Running all 5 approaches x 5 seeds = 25 experiments on Reacher-hard.
Each experiment: collect data -> train -> evaluate (train + test + long-horizon)


In [9]:
"""
Cell: Run All Experiments
Purpose: Execute all approach x seed combinations and save results
"""

all_results = []
summaries = {}

for approach in APPROACHES:
    print(f"\n{'='*70}")
    print(f"APPROACH: {approach}")
    print(f"{'='*70}")

    approach_results = []

    for seed in EXPERIMENT_SEEDS:
        print(f"\n  --- Seed {seed} ---")
        set_seed(seed)

        try:
            # Collect training data
            print(f"  Collecting {NUM_EPISODES} Reacher-hard episodes...")
            train_episodes = collect_episodes("reacher", "hard", NUM_EPISODES, seed)
            print(f"  Collected {len(train_episodes)} valid episodes")

            # Build buffer
            if approach == "superposition":
                buffer = SuperpositionReplayBuffer(capacity=10000)
                for ep in train_episodes:
                    buffer.add(ep)
            else:
                buffer = ReplayBuffer(capacity=10000)
                for ep in train_episodes:
                    buffer.add(ep)

            # Create model
            model = create_model(approach, seed=seed)
            num_params = sum(p.numel() for p in model.parameters())
            print(f"  Model: {num_params:,} parameters")

            # Train
            start_time = time.time()
            if approach == "interference_ensemble":
                history = train_interference_ensemble(model, buffer)
            else:
                history = train_single_model(model, buffer, approach, seed)
            elapsed = time.time() - start_time
            print(f"  Training completed in {elapsed:.1f}s")

            # Evaluate on training data
            eval_buffer = ReplayBuffer()
            for ep in train_episodes[:50]:
                eval_buffer.add(ep)
            train_metrics = evaluate_model(model, eval_buffer, approach)

            # Collect and evaluate on test data (different seed offset)
            print(f"  Collecting test episodes...")
            test_episodes = collect_episodes("reacher", "hard", 50, seed + 10000)
            test_buffer = ReplayBuffer()
            for ep in test_episodes:
                test_buffer.add(ep)
            test_metrics = evaluate_model(model, test_buffer, approach)

            # Long-horizon prediction
            horizon_results = evaluate_long_horizon(model, test_buffer, approach)

            result = {
                "approach": approach,
                "seed": seed,
                "train_obs_mse": train_metrics["obs_mse"],
                "train_reward_mse": train_metrics["reward_mse"],
                "test_obs_mse": test_metrics["obs_mse"],
                "test_reward_mse": test_metrics["reward_mse"],
                "long_horizon": {str(k): v for k, v in horizon_results.items()},
                "final_train_loss": float(history["total"].iloc[-1]),
                "time_seconds": elapsed,
                "num_params": num_params,
            }
            approach_results.append(result)
            all_results.append(result)

            # Save per-seed result
            seed_file = RESULTS_DIR / f"{approach}_seed_{seed}.json"
            with open(seed_file, "w") as f:
                json.dump(result, f, indent=2)
            print(f"  Saved: {seed_file.name}")
            print(
                f"  Train MSE: {result['train_obs_mse']:.6f} | "
                f"Test MSE: {result['test_obs_mse']:.6f} | "
                f"Time: {elapsed:.1f}s"
            )

            # Cleanup
            del model, buffer, eval_buffer, test_buffer
            torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

        except Exception as e:
            print(f"  ERROR: {e}")
            traceback.print_exc()
            all_results.append({
                "approach": approach,
                "seed": seed,
                "error": str(e),
            })

    # Aggregate approach results
    valid = [r for r in approach_results if "error" not in r]
    if valid:
        test_mses = [r["test_obs_mse"] for r in valid]
        train_mses = [r["train_obs_mse"] for r in valid]
        reward_mses = [r["test_reward_mse"] for r in valid]
        times = [r["time_seconds"] for r in valid]

        # Aggregate long-horizon predictions
        long_horizon_means = {}
        for h_key in ["5", "10", "15", "20"]:
            h_vals = [
                r["long_horizon"][h_key]
                for r in valid
                if "long_horizon" in r and r["long_horizon"] and h_key in r["long_horizon"]
            ]
            if h_vals:
                long_horizon_means[h_key] = {
                    "mean": float(np.mean(h_vals)),
                    "std": float(np.std(h_vals)),
                }

        summaries[approach] = {
            "test_obs_mse_mean": float(np.mean(test_mses)),
            "test_obs_mse_std": float(np.std(test_mses)),
            "train_obs_mse_mean": float(np.mean(train_mses)),
            "train_obs_mse_std": float(np.std(train_mses)),
            "test_reward_mse_mean": float(np.mean(reward_mses)),
            "test_reward_mse_std": float(np.std(reward_mses)),
            "time_mean": float(np.mean(times)),
            "time_std": float(np.std(times)),
            "num_params": valid[0]["num_params"],
            "num_seeds": len(valid),
            "long_horizon": long_horizon_means,
        }
        print(f"\n  {approach} Summary: Test MSE = {np.mean(test_mses):.6f} +/- {np.std(test_mses):.6f}")
    else:
        print(f"\n  {approach}: ALL SEEDS FAILED")

print("\n" + "=" * 70)
print("ALL EXPERIMENTS COMPLETE")
print("=" * 70)


APPROACH: baseline

  --- Seed 42 ---
  Collected 200 valid episodes
  Model: 4,736,264 parameters
    Step 0/10000: total=2.6683 recon=2.1661 kl=0.4960 reward=0.0063
    Step 1000/10000: total=0.3867 recon=0.1782 kl=0.1796 reward=0.0288
    Step 2000/10000: total=0.3204 recon=0.1442 kl=0.1731 reward=0.0032
    Step 3000/10000: total=0.3016 recon=0.1237 kl=0.1686 reward=0.0092
    Step 4000/10000: total=0.3464 recon=0.1501 kl=0.1577 reward=0.0385
    Step 5000/10000: total=0.3153 recon=0.1280 kl=0.1784 reward=0.0090
    Step 6000/10000: total=0.3501 recon=0.1381 kl=0.1875 reward=0.0244
    Step 7000/10000: total=0.3223 recon=0.1291 kl=0.1699 reward=0.0233
    Step 8000/10000: total=0.2998 recon=0.1137 kl=0.1617 reward=0.0244
    Step 9000/10000: total=0.3215 recon=0.1393 kl=0.1682 reward=0.0140
  Training completed in 630.5s
  Saved: baseline_seed_42.json
  Train MSE: 0.130693 | Test MSE: 0.129294 | Time: 630.5s

  --- Seed 123 ---
  Collected 200 valid episodes
  Model: 4,736,264 par

---
## Save Complete Results

Save aggregated metrics as `complete_metrics.json` and print final summary.


In [10]:
"""
Cell: Save Complete Results
Purpose: Save aggregated results as complete_metrics.json
"""

complete_metrics = {
    "experiment": "phase2_reacher_hard",
    "environment": {
        "domain": "reacher",
        "task": "hard",
        "obs_dim": OBS_DIM,
        "action_dim": ACTION_DIM,
    },
    "config": {
        "stoch_dim": STOCH_DIM,
        "deter_dim": DETER_DIM,
        "hidden_dim": HIDDEN_DIM,
        "batch_size": BATCH_SIZE,
        "seq_len": SEQ_LEN,
        "num_steps": NUM_STEPS,
        "learning_rate": LEARNING_RATE,
        "kl_weight": KL_WEIGHT,
        "grad_clip": GRAD_CLIP,
        "num_episodes": NUM_EPISODES,
        "seeds": EXPERIMENT_SEEDS,
    },
    "summary": summaries,
    "raw_results": all_results,
}

metrics_file = RESULTS_DIR / "complete_metrics.json"
with open(metrics_file, "w") as f:
    json.dump(complete_metrics, f, indent=2, default=str)
print(f"Saved complete metrics: {metrics_file}")

# Print final summary table
print(f"\n{'='*70}")
print("REACHER-HARD RESULTS SUMMARY")
print(f"{'='*70}")
print(f"{'Approach':<25} {'Test MSE':>15} {'Train MSE':>15} {'Time (s)':>12} {'Params':>10}")
print("-" * 80)
for approach in APPROACHES:
    if approach in summaries:
        s = summaries[approach]
        print(f"{approach:<25} "
              f"{s['test_obs_mse_mean']:.6f}\u00b1{s['test_obs_mse_std']:.4f} "
              f"{s['train_obs_mse_mean']:.6f}\u00b1{s['train_obs_mse_std']:.4f} "
              f"{s['time_mean']:>8.1f}\u00b1{s['time_std']:.1f} "
              f"{s['num_params']:>10,}")
    else:
        print(f"{approach:<25} {'FAILED':>15}")

# Identify best
if summaries:
    best = min(summaries.items(), key=lambda x: x[1]["test_obs_mse_mean"])
    baseline_mse = summaries.get("baseline", {}).get("test_obs_mse_mean", float("nan"))
    if baseline_mse and not np.isnan(baseline_mse):
        improvement = (baseline_mse - best[1]["test_obs_mse_mean"]) / baseline_mse * 100
        print(f"\nBest approach: {best[0]} ({improvement:+.1f}% vs baseline)")

print(f"\nResults saved to: {RESULTS_DIR}")
print("Done. Next: Update 04_reacher_experiments.ipynb to load and analyze these results.")

Saved complete metrics: d:\Git Repos\Quantum-Enhanced-Simulation-Learning-for-Reinforcement-Learning\experiments\results\phase2\reacher_hard\complete_metrics.json

REACHER-HARD RESULTS SUMMARY
Approach                         Test MSE       Train MSE     Time (s)     Params
--------------------------------------------------------------------------------
baseline                  0.126925±0.0044 0.128952±0.0045    613.6±47.6  4,736,264
quantum_tunneling         0.127905±0.0039 0.127366±0.0059    537.8±12.0  4,736,264
superposition             0.904161±0.0207 0.912951±0.0264    837.6±214.6  4,736,264
entanglement              0.128740±0.0036 0.129991±0.0104    704.0±174.1  5,262,216
interference_ensemble     0.067713±0.0030 0.067032±0.0041   4395.2±1263.2 23,681,326

Best approach: interference_ensemble (+46.7% vs baseline)

Results saved to: d:\Git Repos\Quantum-Enhanced-Simulation-Learning-for-Reinforcement-Learning\experiments\results\phase2\reacher_hard
Done. Next: Update 04_reacher_